In [2]:
from pathlib import Path
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.sample import sample_gen

In [3]:


# 📂 Шлях до HAND-растрових файлів
hand_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/hand_outputs")
hand_rasters = sorted(hand_folder.glob("*_hand_2000.tif"))


In [4]:

# 📖 Читання GeoDataFrame з ICESat-2
ice_gdf = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_dems_delta_32635.parquet")

# 🔧 Створюємо координати для вибірки
coords = [(geom.x, geom.y) for geom in ice_gdf.geometry]

In [5]:
# 🔁 Обхід по кожному HAND-растру
for path in hand_rasters:
    name = path.stem.replace("_utm32635_hand_2000", "")
    col_name = f"hand_{name}"

    with rasterio.open(path) as src:
        assert src.crs == ice_gdf.crs, f"❌ CRS не збігається: {src.crs} vs {ice_gdf.crs}"
        sampled = list(sample_gen(src, coords))
        values = [val[0] if val and val[0] != src.nodata else np.nan for val in sampled]

    ice_gdf[col_name] = values
    print(f"✅ Додано: {col_name}")

# 💾 Зберігаємо результат з доданими колонками HAND
output_path = "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_HAND.parquet"
ice_gdf.to_parquet(output_path)
print(f"📦 HAND збережено у: {output_path}")


✅ Додано: hand_alos_dem
✅ Додано: hand_aster_dem
✅ Додано: hand_copernicus_dеm
✅ Додано: hand_fab_dem
✅ Додано: hand_nasa_dem
✅ Додано: hand_srtm_dem
✅ Додано: hand_tan_dem
📦 HAND збережено у: /mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_HAND.parquet
